# ProspectML - Exploratory Data Analysis & Feature Selection

This notebook explores the cleaned ProspectML dataset to identify which Minor League statistics are most useful for predicting future Major League hitting performance.

The objectives of this notebook are to:

- load the cleaned player-level dataset
- inspect the overall structure and quality of the data
- examine Major League outcome variables
- evaluate relationships between Minor League and Major League statistics
- identify candidate features for machine learning models

The results of this notebook will guide feature selection for the baseline predictive model developed in the next stage of the project.

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

## Load Cleaned Dataset

The cleaned ProspectML dataset created in Notebook 1 is loaded for exploration and feature selection.

In [2]:
prospectml_master = pd.read_csv(
    "c:\\Users\\ncodi\\OneDrive\\Documents\\ProspectML\\Data\\prospectml_master.csv"
)

## Inspect Dataset

Before performing any analysis, the dataset is inspected to verify its dimensions, variable types, and overall structure.

In [3]:
prospectml_master.shape
prospectml_master.head()

,Career_#,Name,Career_Age,Career_PA,Career_BB%,Career_K%,Career_BB/K,Career_AVG,Career_OBP,Career_SLG,...,MLB_SLG,MLB_OPS,MLB_ISO,MLB_Spd,MLB_BABIP,MLB_wSB,MLB_wRC,MLB_wRAA,MLB_wOBA,MLB_wRC_plus
0,4148,Paulo Orlando,20-33,5131,5.50%,18.70%,0.30,0.273,0.324,0.408,...,0.384,0.673,0.121,5.8,0.327,0.3,87,-20.1,0.290,78
1,2894,Gorkys Hernández,18-31,5070,8.60%,20.30%,0.42,0.274,0.342,0.379,...,0.351,0.643,0.121,5.8,0.287,0.2,96,-33.1,0.281,74
2,4026,Pedro Florimón,19-34,5048,9.60%,25.40%,0.38,0.250,0.327,0.369,...,0.319,0.588,0.108,6.5,0.287,0.7,61,-35.9,0.262,59
3,2708,Iván De Jesús Jr.,19-32,5018,9.60%,16.20%,0.59,0.297,0.369,0.399,...,0.327,0.630,0.085,2.9,0.317,-1.1,47,-16.1,0.279,71
4,2596,Matt Davidson,18-30,4895,9.40%,26.60%,0.35,0.254,0.336,0.453,...,0.430,0.719,0.209,1.4,0.290,-1.8,123,-8.6,0.308,93


In [4]:
prospectml_master = prospectml_master.drop(
    columns=[
        "Career_#",
        "HighA_#",
        "AA_#",
        "AAA_#",
        "MLB_#"
    ]
)

## Explore MLB Outcome Variables

The Major League statistics represent the target outcomes for this project.

Before identifying predictive Minor League features, the distribution of the MLB performance metrics is examined to better understand the range and variation of player outcomes within the dataset.

In [5]:
mlb_columns = [col for col in prospectml_master.columns if col.startswith("MLB_")]

prospectml_master[mlb_columns].describe()

,MLB_PA,MLB_BB/K,MLB_AVG,MLB_OBP,MLB_SLG,MLB_OPS,MLB_ISO,MLB_Spd,MLB_BABIP,MLB_wSB,MLB_wRC,MLB_wRAA,MLB_wOBA,MLB_wRC_plus
count,936.000000,936.000000,936.000000,936.000000,936.000000,936.000000,936.000000,936.000000,936.000000,936.000000,936.000000,936.000000,936.000000,936.000000
mean,2278.838675,0.379231,0.245296,0.311403,0.392929,0.704342,0.147655,4.224786,0.295229,0.181197,272.967949,5.368162,0.306922,91.355769
std,1770.736292,0.146198,0.022809,0.025534,0.050933,0.068765,0.043874,1.652470,0.024935,5.273466,256.689111,69.593000,0.026865,18.922542
min,500.000000,0.110000,0.172000,0.225000,0.259000,0.504000,0.029000,0.900000,0.194000,-11.900000,25.000000,-189.000000,0.225000,34.000000
25%,950.500000,0.280000,0.231000,0.294000,0.358000,0.660000,0.117000,2.900000,0.280000,-2.300000,94.000000,-27.000000,0.289000,79.000000
50%,1690.000000,0.350000,0.246000,0.312000,0.394000,0.707000,0.148000,4.100000,0.295000,-0.800000,182.500000,-9.850000,0.308000,92.000000
75%,3141.250000,0.460000,0.261000,0.328000,0.426000,0.747000,0.178250,5.400000,0.312000,0.800000,354.250000,13.250000,0.325000,103.000000
max,9707.000000,1.210000,0.317000,0.413000,0.615000,1.028000,0.322000,9.000000,0.371000,36.000000,1600.000000,555.300000,0.425000,178.000000


## Same-Statistic Translation Analysis

The first step in feature evaluation is measuring how well Minor League statistics translate to Major League performance.

Each Minor League statistic is compared against the equivalent Major League statistic to determine which skills carry over most consistently between levels.

In [7]:
percent_columns = [
    col for col in prospectml_master.columns 
    if "%" in col
]

percent_columns

['Career_BB%',
 'Career_K%',
 'HighA_BB%',
 'HighA_K%',
 'AA_BB%',
 'AA_K%',
 'AAA_BB%',
 'AAA_K%',
 'MLB_BB%',
 'MLB_K%']

In [8]:
for col in percent_columns:
    prospectml_master[col] = (
        prospectml_master[col]
        .str.replace("%", "", regex=False)
        .astype(float)
    )

In [9]:
milb_stats = [
    "AVG",
    "OBP",
    "SLG",
    "OPS",
    "ISO",
    "BB%",
    "K%",
    "BB/K",
    "BABIP",
    "wOBA",
    "wRC_plus"
]

In [10]:
levels = ["Career", "HighA", "AA", "AAA"]

translation_results = []

for level in levels:
    for stat in milb_stats:
        milb_col = f"{level}_{stat}"
        mlb_col = f"MLB_{stat}"
        
        if milb_col in prospectml_master.columns and mlb_col in prospectml_master.columns:
            correlation = prospectml_master[milb_col].corr(prospectml_master[mlb_col])
            
            translation_results.append({
                "Level": level,
                "MiLB Stat": stat,
                "MLB Stat": stat,
                "Correlation": correlation
            })

translation_df = pd.DataFrame(translation_results)

translation_df

,Level,MiLB Stat,MLB Stat,Correlation
0,Career,AVG,AVG,0.485135
1,Career,OBP,OBP,0.553573
2,Career,SLG,SLG,0.575036
3,Career,OPS,OPS,0.525992
4,Career,ISO,ISO,0.691208
5,Career,BB%,BB%,0.745300
6,Career,K%,K%,0.792216
7,Career,BB/K,BB/K,0.709986
8,Career,BABIP,BABIP,0.507668
9,Career,wOBA,wOBA,0.542222


In [11]:
translation_df.sort_values(
    by="Correlation",
    ascending=False
)

,Level,MiLB Stat,MLB Stat,Correlation
6,Career,K%,K%,0.792216
5,Career,BB%,BB%,0.745300
39,AAA,K%,K%,0.734745
7,Career,BB/K,BB/K,0.709986
4,Career,ISO,ISO,0.691208
38,AAA,BB%,BB%,0.650666
40,AAA,BB/K,BB/K,0.641403
28,AA,K%,K%,0.637023
10,Career,wRC_plus,wRC_plus,0.588490
2,Career,SLG,SLG,0.575036


In [12]:
translation_pivot = translation_df.pivot(
    index="MiLB Stat",
    columns="Level",
    values="Correlation"
)

translation_pivot

Level,AA,AAA,Career,HighA
MiLB Stat,,,,
AVG,0.259489,0.223652,0.485135,0.165585
BABIP,0.258561,0.249456,0.507668,0.230282
BB%,0.550118,0.650666,0.745300,0.436734
BB/K,0.467343,0.641403,0.709986,0.295746
ISO,0.471272,0.524667,0.691208,0.435295
K%,0.637023,0.734745,0.792216,0.557029
OBP,0.319908,0.299017,0.553573,0.177353
OPS,0.307460,0.291807,0.525992,0.214101
SLG,0.348132,0.362552,0.575036,0.283723


## Initial Skill Translation Findings

The same-stat translation analysis evaluates how consistently Minor League skills carry over to Major League performance.

Most Minor League statistics showed moderate relationships with their Major League equivalents, with only a small number of features exceeding a strong correlation threshold.

The strongest relationships were primarily found in plate discipline and power-related statistics, suggesting that these skills may translate more consistently across levels than other offensive metrics.

Because this analysis only evaluates whether the same skill carries over, additional analysis is required to determine which Minor League statistics best predict overall Major League hitting success.

## Removing Speed Statistics

Because this project focuses on predicting future MLB hitting performance, speed-based statistics are removed from the dataset.

Although speed can contribute to offensive value through baserunning and infield hits, FanGraphs' Spd metric primarily measures baserunning ability rather than hitting skill. Removing these variables keeps the feature set focused on offensive production and hitting-related attributes.

In [14]:
spd_columns = [
    col for col in prospectml_master.columns 
    if "Spd" in col
]

prospectml_master = prospectml_master.drop(columns=spd_columns)

spd_columns

['Career_Spd', 'HighA_Spd', 'AA_Spd', 'AAA_Spd', 'MLB_Spd']

## Feature Evaluation by Major League Outcome

After evaluating how individual skills translate from the Minor Leagues to Major League Baseball, the next step is to identify which Minor League statistics are most useful for predicting important measures of MLB offensive performance.

Rather than evaluating every possible combination of statistics, this analysis focuses on several key MLB outcome metrics that represent different aspects of offensive value, including on-base ability, overall offensive production, power, and advanced run creation.

For each MLB outcome, every relevant Minor League statistic from the Career, High-A, AA, and AAA levels will be compared using Pearson correlation coefficients. These results will help identify the most informative predictor variables for the machine learning models developed later in the project.

In [15]:
# Predictor statistics for each MLB outcome

# Predicting MLB On-Base Percentage (OBP)
obp_predictors = [
    "AVG",
    "OBP",
    "OPS",
    "BB%",
    "K%",
    "BB/K",
    "BABIP",
    "wOBA"
]


# Predicting MLB On-Base Plus Slugging (OPS)
ops_predictors = [
    "AVG",
    "OBP",
    "SLG",
    "OPS",
    "ISO",
    "BB%",
    "K%",
    "BB/K",
    "BABIP",
    "wOBA",
    "wRC_plus"
]


# Predicting MLB Isolated Power (ISO)
iso_predictors = [
    "SLG",
    "OPS",
    "ISO",
    "BABIP",
    "wOBA"
]


# Predicting MLB Weighted On-Base Average (wOBA)
woba_predictors = [
    "AVG",
    "OBP",
    "SLG",
    "OPS",
    "ISO",
    "BB%",
    "K%",
    "BB/K",
    "BABIP",
    "wOBA",
    "wRC_plus"
]


# Predicting MLB Weighted Runs Above Average (wRAA)
wraa_predictors = [
    "AVG",
    "OBP",
    "SLG",
    "OPS",
    "ISO",
    "BB%",
    "K%",
    "BB/K",
    "BABIP",
    "wOBA",
    "wRC_plus"
]


# Predicting MLB Weighted Runs Created (wRC)
wrc_predictors = [
    "AVG",
    "OBP",
    "SLG",
    "OPS",
    "ISO",
    "BB%",
    "K%",
    "BB/K",
    "BABIP",
    "wOBA",
    "wRC_plus"
]


# Predicting MLB Weighted Runs Created Plus (wRC+)
wrcplus_predictors = [
    "AVG",
    "OBP",
    "SLG",
    "OPS",
    "ISO",
    "BB%",
    "K%",
    "BB/K",
    "BABIP",
    "wOBA",
    "wRC_plus"
]

## Correlation Function

To avoid repeating the same analysis for each Major League outcome statistic, a reusable function is created.

Given an MLB target statistic, the function calculates the Pearson correlation between that target and every relevant Minor League statistic across the Career, High-A, AA, and AAA levels. The resulting table is automatically sorted from the strongest positive correlation to the weakest.

In [16]:
def correlation_table(mlb_target, predictors):
    """
    Calculates correlations between selected Minor League predictors
    and a specified MLB outcome statistic.
    """

    results = []

    levels = ["Career", "HighA", "AA", "AAA"]

    for level in levels:
        for stat in predictors:

            milb_col = f"{level}_{stat}"

            if milb_col in prospectml_master.columns:

                correlation = prospectml_master[milb_col].corr(
                    prospectml_master[mlb_target]
                )

                results.append({
                    "MLB Target": mlb_target,
                    "Level": level,
                    "MiLB Feature": milb_col,
                    "Correlation": correlation
                })

    df = (
        pd.DataFrame(results)
        .sort_values(
            by="Correlation",
            ascending=False
        )
        .reset_index(drop=True)
    )

    return df

## MLB OBP (On-Base Percentage)

OPS combines a hitter's ability to reach base, making it one of the most widely used traditional measures of offensive production.

This section evaluates which Minor League statistics are most strongly associated with future MLB OBP. The results will help identify which offensive skills provide the greatest predictive value when projecting future hitters.

In [17]:
obp_correlations = correlation_table(
    "MLB_OBP",
    obp_predictors
)

obp_correlations

,MLB Target,Level,MiLB Feature,Correlation
0,MLB_OBP,Career,Career_OBP,0.553573
1,MLB_OBP,Career,Career_wOBA,0.436843
2,MLB_OBP,Career,Career_AVG,0.384204
3,MLB_OBP,Career,Career_BB/K,0.377300
4,MLB_OBP,Career,Career_OPS,0.375079
5,MLB_OBP,Career,Career_BB%,0.362039
6,MLB_OBP,AAA,AAA_BB/K,0.333764
7,MLB_OBP,AA,AA_OBP,0.319908
8,MLB_OBP,Career,Career_BABIP,0.304194
9,MLB_OBP,AAA,AAA_BB%,0.303235


In [18]:
obp_selected_features = [
    "Career_OBP",
    "Career_wOBA",
    "Career_AVG",
    "Career_BB/K",
    "Career_OPS",
    "Career_BB%"
]

## Selected MLB OBP Predictors

Based on the correlation analysis, six Minor League statistics were selected as potential predictors of future MLB on-base percentage. These features were chosen because they showed a meaningful relationship with MLB OBP, using an absolute correlation threshold of 0.35.

The selected predictors, ranked by correlation strength were:

- Career_OBP
- Career_wOBA
- Career_AVG
- Career_BB/K
- Career_OPS
- Career_BB%

An important finding from this analysis is that all selected predictors were Career-level statistics rather than statistics from a single Minor League level. This suggests that cumulative Minor League performance may provide a more reliable indicator of future MLB on-base ability than performance at an individual developmental stage.

The selected features represent multiple aspects of on-base skill, including:
- overall ability to reach base (OBP, OPS)
- contact ability (AVG)
- plate discipline (BB%, BB/K)
- overall offensive quality (wOBA)

These features will be considered as potential predictors in the final model, while remaining Minor League statistics will continue to be evaluated against other MLB offensive outcomes.

## MLB OPS Predictor Analysis

After evaluating predictors of MLB on-base percentage, the next step is analyzing which Minor League statistics are most associated with future MLB OPS.

OPS combines a player's ability to reach base and hit for power, making it a broad measure of offensive production. This analysis evaluates which Minor League offensive statistics provide the strongest relationship with future MLB OPS.

The same correlation-based feature evaluation process is used, with predictors considered based on the strength of their relationship with MLB OPS. Features with meaningful correlations will be selected as potential inputs for future modeling.

In [19]:
ops_correlations = correlation_table(
    "MLB_OPS",
    ops_predictors
)

ops_correlations

,MLB Target,Level,MiLB Feature,Correlation
0,MLB_OPS,Career,Career_wRC_plus,0.549909
1,MLB_OPS,Career,Career_wOBA,0.532576
2,MLB_OPS,Career,Career_OPS,0.525992
3,MLB_OPS,Career,Career_SLG,0.508085
4,MLB_OPS,Career,Career_ISO,0.430363
5,MLB_OPS,Career,Career_OBP,0.366348
6,MLB_OPS,Career,Career_AVG,0.331341
7,MLB_OPS,AA,AA_SLG,0.330846
8,MLB_OPS,AA,AA_ISO,0.329258
9,MLB_OPS,AAA,AAA_wRC_plus,0.316682


In [20]:
ops_selected_features = [
    "Career_wRC_plus",
    "Career_wOBA",
    "Career_OPS",
    "Career_SLG",
    "Career_ISO",
    "Career_OBP"
]

## Selected MLB OPS Predictors

Based on the correlation analysis, six Minor League statistics were selected as potential predictors of future MLB OPS. These features were chosen because they demonstrated a meaningful relationship with MLB OPS using an absolute correlation threshold of 0.35.

The selected predictors, ranked by correlation strength, were:

- Career_wRC_plus
- Career_wOBA
- Career_OPS
- Career_SLG
- Career_ISO
- Career_OBP

All selected predictors were Career-level statistics, indicating that cumulative Minor League offensive performance provided the strongest relationship with future MLB OPS compared to individual Minor League levels.

Career_wRC_plus was the strongest predictor of MLB OPS, suggesting that adjusted offensive performance is highly associated with future overall offensive production. Career_wOBA and Career_OPS also ranked highly, indicating that overall offensive quality and consistent production are important indicators when projecting future MLB OPS.

Career_SLG and Career_ISO were also selected, highlighting the importance of power production when projecting OPS. Career_OBP contributed additional information by representing a player's ability to consistently reach base.

The selected features represent several components of offensive production:
- adjusted offensive value (wRC_plus)
- overall offensive quality (wOBA)
- overall production (OPS)
- power production (SLG, ISO)
- ability to reach base (OBP)

These features will be considered as potential predictors in the final model, while remaining Minor League statistics will continue to be evaluated against other MLB offensive outcomes.

## MLB ISO Predictor Analysis

After evaluating predictors of MLB on-base percentage and OPS, the next step is analyzing which Minor League statistics are most associated with future MLB isolated power (ISO).

ISO measures a hitter's ability to produce extra-base hits by isolating power production from batting average. Because ISO focuses specifically on power, this analysis emphasizes Minor League statistics related to extra-base hit ability, quality of contact, and overall offensive impact.

The same correlation-based feature evaluation process is used, with predictors considered based on their relationship with MLB ISO. Features with meaningful correlations will be selected as potential inputs for future modeling.

In [22]:
iso_correlations = correlation_table(
    "MLB_ISO",
    iso_predictors
)

iso_correlations

,MLB Target,Level,MiLB Feature,Correlation
0,MLB_ISO,Career,Career_ISO,0.691208
1,MLB_ISO,Career,Career_SLG,0.626229
2,MLB_ISO,Career,Career_OPS,0.536336
3,MLB_ISO,AAA,AAA_ISO,0.524667
4,MLB_ISO,Career,Career_wOBA,0.499793
5,MLB_ISO,AA,AA_ISO,0.471272
6,MLB_ISO,HighA,HighA_ISO,0.435295
7,MLB_ISO,AAA,AAA_SLG,0.422620
8,MLB_ISO,AAA,AAA_OPS,0.357759
9,MLB_ISO,AA,AA_SLG,0.351695


In [23]:
iso_selected_features = [
    "Career_ISO",
    "Career_SLG",
    "Career_OPS",
    "AAA_ISO",
    "Career_wOBA",
    "AA_ISO",
    "HighA_ISO",
    "AAA_SLG",
    "AAA_OPS",
    "AA_SLG"
]

## Selected MLB ISO Predictors

Based on the correlation analysis, ten Minor League statistics were selected as potential predictors of future MLB isolated power (ISO). These features were chosen because they demonstrated a meaningful relationship with MLB ISO using an absolute correlation threshold of 0.35.

The selected predictors, ranked by correlation strength, were:

- Career_ISO
- Career_SLG
- Career_OPS
- AAA_ISO
- Career_wOBA
- AA_ISO
- HighA_ISO
- AAA_SLG
- AAA_OPS
- AA_SLG

Unlike the previous OBP and OPS analyses, this outcome included predictors from multiple Minor League levels. While Career-level statistics remained the strongest predictors, individual-level power statistics from AAA, AA, and High-A also demonstrated meaningful relationships with future MLB ISO.

The strongest predictor was Career_ISO, which suggests that a player's accumulated Minor League power profile is the most reliable indicator of future MLB isolated power. Career_SLG and Career_OPS also ranked highly, reinforcing the importance of overall extra-base hit production.

The inclusion of AA and AAA ISO statistics suggests that evaluating power development at advanced Minor League levels may provide additional predictive value. This may indicate that a player's power output closer to reaching the MLB level is an important factor when projecting future power production.

The selected features represent several components related to power development:
- direct power production (ISO)
- extra-base hit ability (SLG)
- overall offensive impact (OPS, wOBA)

These features will be considered as potential predictors in the final model, while remaining Minor League statistics will continue to be evaluated against other MLB offensive outcomes.

## MLB wOBA Predictor Analysis

After evaluating predictors of MLB on-base percentage, OPS, and ISO, the next step is analyzing which Minor League statistics are most associated with future MLB weighted on-base average (wOBA).

wOBA is an overall offensive metric that assigns value to different offensive outcomes based on their contribution to run creation. Unlike traditional statistics, wOBA captures a combination of hitting ability, plate discipline, and power production.

This analysis evaluates which Minor League offensive statistics provide the strongest relationship with future MLB wOBA. The same correlation-based feature selection process is used, with predictors considered based on their strength of relationship with MLB wOBA.

Features with meaningful correlations will be selected as potential predictors for future modeling.

In [24]:
woba_correlations = correlation_table(
    "MLB_wOBA",
    woba_predictors
)

woba_correlations

,MLB Target,Level,MiLB Feature,Correlation
0,MLB_wOBA,Career,Career_wRC_plus,0.554313
1,MLB_wOBA,Career,Career_wOBA,0.542222
2,MLB_wOBA,Career,Career_OPS,0.526065
3,MLB_wOBA,Career,Career_SLG,0.480486
4,MLB_wOBA,Career,Career_OBP,0.423752
5,MLB_wOBA,Career,Career_ISO,0.394120
6,MLB_wOBA,Career,Career_AVG,0.342431
7,MLB_wOBA,AA,AA_SLG,0.328708
8,MLB_wOBA,AA,AA_OPS,0.323140
9,MLB_wOBA,AA,AA_wRC_plus,0.322915


In [25]:
woba_selected_features = [
    "Career_wRC_plus",
    "Career_wOBA",
    "Career_OPS",
    "Career_SLG",
    "Career_OBP",
    "Career_ISO"
]

## Selected MLB wOBA Predictors

Based on the correlation analysis, six Minor League statistics were selected as potential predictors of future MLB weighted on-base average (wOBA). These features were chosen because they demonstrated a meaningful relationship with MLB wOBA using an absolute correlation threshold of 0.35.

The selected predictors, ranked by correlation strength, were:

- Career_wRC_plus
- Career_wOBA
- Career_OPS
- Career_SLG
- Career_OBP
- Career_ISO

All selected predictors were Career-level statistics, indicating that cumulative Minor League offensive performance provided the strongest relationship with future MLB wOBA compared to individual Minor League levels.

Career_wRC_plus was the strongest predictor of MLB wOBA, suggesting that adjusted offensive performance provides a strong indication of future offensive value. Career_wOBA and Career_OPS also ranked highly, reinforcing the importance of overall offensive quality and consistent production when projecting future MLB wOBA.

Career_SLG and Career_ISO were selected, highlighting the importance of power production, while Career_OBP contributed information about a player's ability to consistently reach base.

The selected features represent several components of offensive performance:
- adjusted offensive value (wRC_plus)
- overall offensive quality (wOBA)
- overall production (OPS)
- power production (SLG, ISO)
- ability to reach base (OBP)

These features will be considered as potential predictors in the final model, while remaining Minor League statistics will continue to be evaluated against other MLB offensive outcomes.

## MLB wRAA Predictor Analysis

After evaluating predictors of MLB on-base percentage, OPS, ISO, and wOBA, the next step is analyzing which Minor League statistics are most associated with future MLB weighted runs above average (wRAA).

wRAA measures a player's offensive contribution compared to league average, incorporating multiple aspects of hitting performance including getting on base and producing power. Because wRAA represents overall offensive value, this analysis evaluates whether Minor League statistics that measure offensive production can help predict future MLB offensive contribution.

The same correlation-based feature selection process is used, with predictors evaluated based on their relationship with MLB wRAA. Features with meaningful correlations will be selected as potential inputs for future modeling.

In [26]:
wraa_correlations = correlation_table(
    "MLB_wRAA",
    wraa_predictors
)

wraa_correlations

,MLB Target,Level,MiLB Feature,Correlation
0,MLB_wRAA,Career,Career_wRC_plus,0.486036
1,MLB_wRAA,Career,Career_wOBA,0.480528
2,MLB_wRAA,Career,Career_OPS,0.467893
3,MLB_wRAA,Career,Career_SLG,0.427609
4,MLB_wRAA,Career,Career_OBP,0.376775
5,MLB_wRAA,Career,Career_ISO,0.345268
6,MLB_wRAA,Career,Career_AVG,0.317138
7,MLB_wRAA,Career,Career_BABIP,0.260501
8,MLB_wRAA,AA,AA_SLG,0.255190
9,MLB_wRAA,AA,AA_ISO,0.255092


In [27]:
wraa_selected_features = [
    "Career_wRC_plus",
    "Career_wOBA",
    "Career_OPS",
    "Career_SLG",
    "Career_OBP"
]

## Selected MLB wRAA Predictors

Based on the correlation analysis, five Minor League statistics were selected as potential predictors of future MLB weighted runs above average (wRAA). These features were chosen because they demonstrated a meaningful relationship with MLB wRAA using an absolute correlation threshold of 0.35.

The selected predictors, ranked by correlation strength, were:

- Career_wRC_plus
- Career_wOBA
- Career_OPS
- Career_SLG
- Career_OBP

All selected predictors were Career-level statistics, indicating that cumulative Minor League offensive performance provided the strongest relationship with future MLB offensive contribution compared to individual Minor League levels.

Career_wRC_plus was the strongest predictor of MLB wRAA, suggesting that adjusted offensive performance is highly associated with future offensive contribution above league average. Career_wOBA and Career_OPS also ranked highly, showing that overall offensive quality and consistent production are important indicators when projecting future MLB offensive value.

Career_SLG and Career_OBP were also selected, highlighting the importance of both power production and the ability to consistently reach base when evaluating future offensive contribution.

The selected features represent several components of offensive performance:
- adjusted offensive value (wRC_plus)
- overall offensive quality (wOBA)
- overall production (OPS)
- power production (SLG)
- ability to reach base (OBP)

Counting statistics such as wRAA and wRC were excluded as predictors because they are influenced by playing time and opportunity. This analysis prioritizes rate statistics that better measure a player's underlying offensive ability.

These features will be considered as potential predictors in the final model, while remaining Minor League statistics will continue to be evaluated against other MLB offensive outcomes.

## MLB wRC Predictor Analysis

After evaluating predictors of MLB on-base percentage, OPS, ISO, wOBA, and wRAA, the next step is analyzing which Minor League statistics are most associated with future MLB weighted runs created (wRC).

wRC measures a player's total offensive contribution by estimating the number of runs created through their offensive performance. Although wRC is a counting statistic, this analysis focuses on identifying the underlying hitting skills that contribute to future offensive production.

Minor League counting statistics such as wRC and wRAA are excluded as predictors because they are heavily influenced by playing time and opportunity. Instead, this analysis prioritizes rate statistics that measure a player's offensive ability independent of volume.

The same correlation-based feature selection process is used, with predictors evaluated based on their relationship with MLB wRC. Features with meaningful correlations will be selected as potential inputs for future modeling.

In [28]:
wrc_correlations = correlation_table(
    "MLB_wRC",
    wrc_predictors
)

wrc_correlations

,MLB Target,Level,MiLB Feature,Correlation
0,MLB_wRC,Career,Career_wOBA,0.382934
1,MLB_wRC,Career,Career_wRC_plus,0.377547
2,MLB_wRC,Career,Career_OPS,0.367057
3,MLB_wRC,Career,Career_AVG,0.362401
4,MLB_wRC,Career,Career_OBP,0.330380
5,MLB_wRC,Career,Career_SLG,0.318617
6,MLB_wRC,Career,Career_BABIP,0.253739
7,MLB_wRC,Career,Career_ISO,0.201645
8,MLB_wRC,AA,AA_OPS,0.190393
9,MLB_wRC,AA,AA_wOBA,0.189623


In [29]:
wrc_selected_features = [
    "Career_wOBA",
    "Career_wRC_plus",
    "Career_OPS",
    "Career_AVG"
]

## Selected MLB wRC Predictors

Based on the correlation analysis, four Minor League statistics were selected as potential predictors of future MLB weighted runs created (wRC). These features were chosen because they demonstrated a meaningful relationship with MLB wRC using an absolute correlation threshold of 0.35.

The selected predictors, ranked by correlation strength, were:

- Career_wOBA
- Career_wRC_plus
- Career_OPS
- Career_AVG

All selected predictors were Career-level statistics, indicating that cumulative Minor League offensive performance provided the strongest relationship with future MLB offensive contribution compared to individual Minor League levels.

Career_wOBA was the strongest predictor of MLB wRC, suggesting that overall offensive quality is the most important indicator of future offensive production. Career_wRC_plus and Career_OPS also ranked highly, reinforcing the importance of adjusted offensive value and overall offensive performance when projecting future MLB run creation.

Career_AVG was also selected, showing that contact ability contributes additional information when evaluating future offensive production.

The selected features represent several components of offensive performance:
- overall offensive quality (wOBA)
- adjusted offensive value (wRC_plus)
- overall production (OPS)
- contact ability (AVG)

Counting statistics such as Minor League wRC and wRAA were excluded as predictors because they are influenced by playing time and opportunity. This analysis prioritizes rate statistics that better measure a player's underlying offensive ability rather than accumulated production.

These features will be considered as potential predictors in the final model, while remaining Minor League statistics will continue to be evaluated against other MLB offensive outcomes.

## MLB wRC+ Predictor Analysis

After evaluating predictors of MLB on-base percentage, OPS, ISO, wOBA, wRAA, and wRC, the final step is analyzing which Minor League statistics are most associated with future MLB weighted runs created plus (wRC+).

wRC+ is an adjusted offensive metric that measures a player's offensive production compared to league average while accounting for external factors such as league and park effects. Because wRC+ represents overall offensive quality rather than raw production, it provides a useful measurement of a player's true offensive ability.

This analysis evaluates which Minor League statistics provide the strongest relationship with future MLB wRC+. The same correlation-based feature selection process is used, with predictors evaluated based on their relationship with MLB wRC+.

Minor League counting statistics such as wRC and wRAA are excluded as predictors because they are influenced by playing time and opportunity. Instead, this analysis focuses on rate statistics that better represent a player's underlying hitting ability.

In [30]:
wrcplus_correlations = correlation_table(
    "MLB_wRC_plus",
    wrcplus_predictors
)

wrcplus_correlations

,MLB Target,Level,MiLB Feature,Correlation
0,MLB_wRC_plus,Career,Career_wRC_plus,0.588490
1,MLB_wRC_plus,Career,Career_wOBA,0.548981
2,MLB_wRC_plus,Career,Career_OPS,0.521394
3,MLB_wRC_plus,Career,Career_SLG,0.472365
4,MLB_wRC_plus,Career,Career_OBP,0.427920
5,MLB_wRC_plus,Career,Career_ISO,0.392959
6,MLB_wRC_plus,AAA,AAA_wRC_plus,0.340747
7,MLB_wRC_plus,Career,Career_AVG,0.324303
8,MLB_wRC_plus,AA,AA_SLG,0.319945
9,MLB_wRC_plus,AA,AA_ISO,0.319659


In [32]:
wrcplus_selected_features = [
    "Career_wRC_plus",
    "Career_wOBA",
    "Career_OPS",
    "Career_SLG",
    "Career_OBP",
    "Career_ISO"
]

## Selected MLB wRC+ Predictors

Based on the correlation analysis, six Minor League statistics were selected as potential predictors of future MLB weighted runs created plus (wRC+). These features were chosen because they demonstrated a meaningful relationship with MLB wRC+ using an absolute correlation threshold of 0.35.

The selected predictors, ranked by correlation strength, were:

- Career_wRC_plus
- Career_wOBA
- Career_OPS
- Career_SLG
- Career_OBP
- Career_ISO

All selected predictors were Career-level statistics, indicating that cumulative Minor League offensive performance provided the strongest relationship with future MLB wRC+ compared to individual Minor League levels.

Career_wRC_plus was the strongest predictor of MLB wRC+, which is expected because both statistics measure adjusted offensive performance relative to league average. Career_wOBA and Career_OPS also ranked highly, suggesting that overall offensive quality and consistent production are important indicators when projecting future MLB offensive value.

Career_SLG and Career_ISO were selected, highlighting the importance of power production when projecting future offensive performance. Career_OBP contributed additional information by measuring a player's ability to consistently reach base.

The selected features represent several components of offensive performance:
- adjusted offensive value (wRC_plus)
- overall offensive quality (wOBA)
- overall production (OPS)
- power production (SLG, ISO)
- ability to reach base (OBP)

These features will be considered as potential predictors in the final model, while remaining Minor League statistics will continue to be evaluated against other MLB offensive outcomes.

## Combining Selected Predictors

After evaluating Minor League predictors for each MLB offensive outcome, the next step is to combine all selected features into one master predictor list.

Each MLB outcome identified slightly different predictors, but many features appeared across multiple analyses. Combining these lists allows us to create a complete pool of potential predictors for the final model while preserving all features that demonstrated a meaningful relationship with at least one MLB outcome.

Duplicate features will be removed after combining the lists so that each predictor appears only once in the final feature pool.

In [33]:
# Features selected through correlation analysis
all_selected_features = (
    obp_selected_features +
    ops_selected_features +
    iso_selected_features +
    woba_selected_features +
    wraa_selected_features +
    wrc_selected_features +
    wrcplus_selected_features
)

# Always retain plate discipline features
plate_discipline_features = [
    "Career_BB%",
    "Career_K%",
    "HighA_BB%",
    "HighA_K%",
    "AA_BB%",
    "AA_K%",
    "AAA_BB%",
    "AAA_K%"
]

# Always retain Minor League age features
age_features = [
    "Career_Age",
    "HighA_Age",
    "AA_Age",
    "AAA_Age"
]

# Add plate discipline and age features
all_selected_features += (
    plate_discipline_features +
    age_features
)

# Remove duplicates
all_selected_features = list(set(all_selected_features))

print(all_selected_features)

['HighA_K%', 'AA_ISO', 'HighA_BB%', 'AA_K%', 'Career_OBP', 'Career_BB/K', 'AAA_BB%', 'Career_wRC_plus', 'Career_K%', 'AAA_OPS', 'AA_Age', 'Career_wOBA', 'AAA_K%', 'AA_SLG', 'Career_Age', 'AAA_Age', 'AAA_ISO', 'Career_SLG', 'HighA_Age', 'Career_AVG', 'Career_OPS', 'Career_ISO', 'AAA_SLG', 'AA_BB%', 'Career_BB%', 'HighA_ISO']


## Relationships Between MLB Offensive Statistics

Before creating a single measure of overall offensive ability, it is important to understand how the selected MLB offensive statistics relate to one another.

Many advanced offensive metrics are designed to measure similar aspects of hitting performance, while others capture more specialized skills such as power or on-base ability. Examining the correlations between these statistics helps identify which metrics provide unique information and which may be largely redundant.

The results of this analysis will be used to determine which statistics should be considered when constructing an overall measure of offensive ability.

In [35]:
mlb_stats = [
    "MLB_OBP",
    "MLB_OPS",
    "MLB_ISO",
    "MLB_wOBA",
    "MLB_wRAA",
    "MLB_wRC",
    "MLB_wRC_plus"
]

In [36]:
mlb_corr = prospectml_master[mlb_stats].corr()

mlb_corr.round(3)

,MLB_OBP,MLB_OPS,MLB_ISO,MLB_wOBA,MLB_wRAA,MLB_wRC,MLB_wRC_plus
MLB_OBP,1.000,0.793,0.299,0.857,0.675,0.614,0.825
MLB_OPS,0.793,1.000,0.773,0.990,0.777,0.694,0.963
MLB_ISO,0.299,0.773,1.000,0.713,0.577,0.456,0.717
MLB_wOBA,0.857,0.990,0.713,1.000,0.773,0.686,0.970
MLB_wRAA,0.675,0.777,0.577,0.773,1.000,0.779,0.760
MLB_wRC,0.614,0.694,0.456,0.686,0.779,1.000,0.669
MLB_wRC_plus,0.825,0.963,0.717,0.970,0.760,0.669,1.000


In [40]:
kept_mlb_stats = [
    "MLB_OBP",
    "MLB_ISO",
    "MLB_wOBA",
    "MLB_wRC_plus"
]

In [41]:
final_selected_features = sorted(
    list(set(all_selected_features))
)

print(final_selected_features)

['AAA_Age', 'AAA_BB%', 'AAA_ISO', 'AAA_K%', 'AAA_OPS', 'AAA_SLG', 'AA_Age', 'AA_BB%', 'AA_ISO', 'AA_K%', 'AA_SLG', 'Career_AVG', 'Career_Age', 'Career_BB%', 'Career_BB/K', 'Career_ISO', 'Career_K%', 'Career_OBP', 'Career_OPS', 'Career_SLG', 'Career_wOBA', 'Career_wRC_plus', 'HighA_Age', 'HighA_BB%', 'HighA_ISO', 'HighA_K%']


In [42]:
keep_columns = (
    ["PlayerID"] +
    final_selected_features +
    kept_mlb_stats
)

prospectml_model = prospectml_master[keep_columns].copy()

## Final MLB Hitting Ability Metrics

After evaluating the relationships between MLB offensive statistics, the final MLB metrics were selected to represent overall hitting ability.

The objective of this project is to evaluate a player's underlying offensive skill rather than their total accumulated production or individual components that are already captured by broader offensive metrics.

Several statistics were removed:

- Counting statistics such as PA were removed because they represent opportunity and playing time rather than pure hitting ability.
- wRC and wRAA were removed because they measure total offensive contribution and are heavily influenced by playing time.
- OPS was removed because it was highly correlated with other offensive metrics and overlaps with OBP and power-based measurements.
- AVG, SLG, BB%, K%, BB/K, and BABIP were removed because their information is largely captured through more comprehensive offensive metrics.
- wSB was removed because it measures baserunning value rather than hitting performance.
- MiLB stats that were not picked as predictors were also removed

The final MLB metrics selected were:

- MLB_OBP: Represents a player's ability to consistently reach base.
- MLB_ISO: Represents a player's power ability independent of batting average.
- MLB_wOBA: Represents overall offensive value by weighting offensive events based on their actual run contribution.
- MLB_wRC_plus: Represents adjusted offensive performance while accounting for league and park effects.

Together, these metrics capture multiple dimensions of hitting ability:
- ability to reach base
- power production
- overall offensive value
- adjusted offensive performance

These metrics will serve as the foundation for creating an overall measure of MLB hitting ability in future analysis.

In [43]:
prospectml_model.head()

,PlayerID,AAA_Age,AAA_BB%,AAA_ISO,AAA_K%,AAA_OPS,AAA_SLG,AA_Age,AA_BB%,AA_ISO,...,Career_wOBA,Career_wRC_plus,HighA_Age,HighA_BB%,HighA_ISO,HighA_K%,MLB_OBP,MLB_ISO,MLB_wOBA,MLB_wRC_plus
0,paulo_orlando,25-33,6.2,0.136,18.0,0.733,0.408,24-31,6.3,0.144,...,0.328,92,21-23,4.6,0.129,20.2,0.289,0.121,0.290,78
1,gorkys_hernandez,23-31,9.4,0.113,22.1,0.722,0.380,21-22,7.1,0.073,...,0.330,102,20-20,10.3,0.123,16.9,0.292,0.121,0.281,74
2,pedro_florimon,25-34,10.1,0.131,27.1,0.715,0.384,22-31,9.2,0.107,...,0.318,93,22-31,8.5,0.152,21.9,0.270,0.108,0.262,59
3,ivan_de_jesus_jr,23-32,8.3,0.112,17.2,0.775,0.412,21-32,13.8,0.102,...,0.351,103,20-20,11.4,0.093,12.7,0.303,0.085,0.279,71
4,matt_davidson,22-30,9.3,0.213,28.7,0.783,0.458,21-21,12.0,0.208,...,0.350,104,19-20,9.3,0.178,24.9,0.290,0.209,0.308,93


In [45]:
prospectml_model.to_csv("C:\\Users\\ncodi\\OneDrive\\Documents\\ProspectML\\Data\\prospectml_model_data.csv", index=False)